In [43]:
import os
import numpy as np
import pandas as pd
import pickle
import time
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim

from torch_geometric.data import Dataset
from torch_geometric.data import Data, HeteroData
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from torch_geometric.nn import global_mean_pool, global_max_pool, global_add_pool

from torch_geometric.utils import scatter

import time
import wandb
from tqdm import tqdm
from collections import Counter
import json
import matplotlib.pyplot as plt
import random

import sys

from pyg_dataset import NetlistDataset

sys.path.append("models/layers/")
from models.model_att import GNN_node
from sklearn.metrics import accuracy_score, precision_score, recall_score


In [58]:


# Function to compute accuracy, precision, and recall
def compute_metrics(true_labels, predicted_labels):
    # Accuracy
    accuracy = accuracy_score(true_labels, predicted_labels)
    
    # Precision
    precision = precision_score(true_labels, predicted_labels, average='binary')
    
    # Recall
    recall = recall_score(true_labels, predicted_labels, average='binary')
    
    return accuracy, precision, recall

### hyperparameter ###
test = False # if only test but not train
restart = False # if restart training
reload_dataset = False # if reload already processed h_dataset

if test:
    restart = True

prediction = 'demand' # one of ['congestion', 'demand']
model_type = "dehnn" #this can be one of ["dehnn", "dehnn_att", "digcn", "digat"] "dehnn_att" might need large memory usage
num_layer = 3 #large number will cause OOM
num_dim = 16 #large number will cause OOM
vn = True #use virtual node or not
trans = False #use transformer or not
aggr = "add" #use aggregation as one of ["add", "max"]
device = "cpu" #use cuda or cpu
learning_rate = 0.001
num_epochs = 200

if not reload_dataset:
    dataset = NetlistDataset(data_dir="../data/superblue", load_pe = True, pl = True, processed = reload_dataset, load_indices=None)
    h_dataset = []
    for data in tqdm(dataset):
        num_instances = data.node_features.shape[0]
        data.num_instances = num_instances
        data.edge_index_sink_to_net[1] = data.edge_index_sink_to_net[1] - num_instances
        data.edge_index_source_to_net[1] = data.edge_index_source_to_net[1] - num_instances
        
        out_degrees = data.net_features[:, 0]
        mask = (out_degrees < 3000)
        mask_edges = mask[data.edge_index_source_to_net[1]] 
        filtered_edge_index_source_to_net = data.edge_index_source_to_net[:, mask_edges]
        data.edge_index_source_to_net = filtered_edge_index_source_to_net

        mask_edges = mask[data.edge_index_sink_to_net[1]] 
        filtered_edge_index_sink_to_net = data.edge_index_sink_to_net[:, mask_edges]
        data.edge_index_sink_to_net = filtered_edge_index_sink_to_net

        h_data = HeteroData()
        h_data['node'].x = data.node_features
        h_data['net'].x = data.net_features.float()

        
        edge_index = torch.concat([data.edge_index_sink_to_net, data.edge_index_source_to_net], dim=1)
        h_data['node', 'to', 'net'].edge_index, h_data['node', 'to', 'net'].edge_weight = gcn_norm(edge_index, add_self_loops=False)
        h_data['node', 'to', 'net'].edge_type = torch.concat([torch.zeros(data.edge_index_sink_to_net.shape[1]), torch.ones(data.edge_index_source_to_net.shape[1])]).bool()
        h_data['net', 'to', 'node'].edge_index, h_data['net', 'to', 'node'].edge_weight = gcn_norm(edge_index.flip(0), add_self_loops=False)
        
        h_data['design_name'] = data['design_name']
        h_data.num_instances = data.node_features.shape[0]
        variant_data_lst = []

        node_congestion = data.node_congestion.long()
        net_congestion = data.net_congestion.long()
        node_demand = data.node_demand.float()
        net_demand = data.net_demand.float()
        node_demand = (node_demand - torch.mean(node_demand)) / torch.std(node_demand)
        net_demand = (net_demand - torch.mean(net_demand))/ torch.std(net_demand)
        
        batch = data.batch
        num_vn = len(np.unique(batch))
        vn_node = torch.concat([global_mean_pool(h_data['node'].x, batch), 
                global_max_pool(h_data['node'].x, batch)], dim=1)

        variant_data_lst.append((node_demand, net_demand, node_congestion, net_congestion, batch, num_vn, vn_node)) 
        h_data['variant_data_lst'] = variant_data_lst
        h_dataset.append(h_data)
        
    torch.save(h_dataset, "h_dataset.pt")
    
else:
    dataset = torch.load("h_dataset.pt")
    h_dataset = []
    for data in dataset:
        h_dataset.append(data)
    
sys.path.append("models/layers/")



100%|███████████████████████████████████████████| 12/12 [00:02<00:00,  4.14it/s]


In [59]:
list1 = [] # less than -2
list2 = [] # between -2 and 0
list3 = [] # between 0 and 3
list4 = [] # greater than 3
copy_dataset = h_dataset.copy()

In [60]:
for i in range(10):
    for j in range(h_dataset[i]["variant_data_lst"][0][0].size(0)):
        val = h_dataset[i]["variant_data_lst"][0][0][j]
        if val < -2:
            list1.append((i, j))
        elif val >= -2 and val < 0:
            list2.append((i, j))
        elif val >= 0 and val < 2:
            list3.append((i, j))
        else:
            list4.append((i, j))

smallest_amt = min(len(list1), len(list2), len(list3), len(list4))
smallest_amt


103049

In [61]:
list1down = random.sample(list1, smallest_amt)
list2down = random.sample(list2, smallest_amt)
list3down = random.sample(list3, smallest_amt)
list4down = random.sample(list4, smallest_amt)

In [62]:
dict_node = {}
for tup in list1down:
    if tup[0] in dict_node:
        dict_node[tup[0]].append(tup[1])
    else:
        dict_node[tup[0]] = [tup[1]]
for tup in list2down:
    if tup[0] in dict_node:
        dict_node[tup[0]].append(tup[1])
    else:
        dict_node[tup[0]] = [tup[1]]
for tup in list3down:
    if tup[0] in dict_node:
        dict_node[tup[0]].append(tup[1])
    else:
        dict_node[tup[0]] = [tup[1]]
for tup in list4down:
    if tup[0] in dict_node:
        dict_node[tup[0]].append(tup[1])
    else:
        dict_node[tup[0]] = [tup[1]]

In [63]:
list1 = [] # less than -2
list2 = [] # between -2 and 0
list3 = [] # between 0 and 2
list4 = [] # greater than 2
for i in range(10):
    for j in range(h_dataset[i]["variant_data_lst"][0][1].size(0)):
        val = h_dataset[i]["variant_data_lst"][0][1][j]
        if val < -2:
            list1.append((i, j))
        elif val >= -2 and val < 0:
            list2.append((i, j))
        elif val >= 0 and val < 2:
            list3.append((i, j))
        else:
            list4.append((i, j))

smallest_amt = min(len(list1), len(list2), len(list3), len(list4))
smallest_amt


187189

In [64]:
list1down = random.sample(list1, smallest_amt)
list2down = random.sample(list2, smallest_amt)
list3down = random.sample(list3, smallest_amt)
list4down = random.sample(list4, smallest_amt)

In [65]:
dict_net = {}
for tup in list1down:
    if tup[0] in dict_net:
        dict_net[tup[0]].append(tup[1])
    else:
        dict_net[tup[0]] = [tup[1]]
for tup in list2down:
    if tup[0] in dict_net:
        dict_net[tup[0]].append(tup[1])
    else:
        dict_net[tup[0]] = [tup[1]]
for tup in list3down:
    if tup[0] in dict_net:
        dict_net[tup[0]].append(tup[1])
    else:
        dict_net[tup[0]] = [tup[1]]
for tup in list4down:
    if tup[0] in dict_net:
        dict_net[tup[0]].append(tup[1])
    else:
        dict_net[tup[0]] = [tup[1]]

In [66]:
# change h_dataset
for i in range(10):
    print(i)
    copy_dataset[i]["num_instances"] = len(dict_node[i])
    copy_dataset[i]["node"]["x"] = h_dataset[i]["node"]["x"][dict_node[i]]
    vdl4 = h_dataset[i]["variant_data_lst"][0][4][dict_node[i]]
    vdl0 = h_dataset[i]["variant_data_lst"][0][0][dict_node[i]]
    vdl2 = h_dataset[i]["variant_data_lst"][0][2][dict_node[i]]
    
    copy_dataset[i]["net"]["x"] = h_dataset[i]["net"]["x"][dict_net[i]]
    vdl1 = h_dataset[i]["variant_data_lst"][0][1][dict_net[i]]
    vdl3 = h_dataset[i]["variant_data_lst"][0][3][dict_net[i]]
    vdl = (vdl0,vdl1, vdl2,vdl3,vdl4,h_dataset[i]["variant_data_lst"][0][5],h_dataset[i]["variant_data_lst"][0][6])
    copy_dataset[i]["variant_data_lst"][0]=vdl

0
1
2
3
4
5
6
7
8
9


In [67]:
for i in range(10):
    # Process ('node', 'to', 'net') edges
    edge_index = h_dataset[i]['node', 'to', 'net']["edge_index"]  # (2, num_edges)
    
    # Convert sets to tensors (without sorting)
    node_set = torch.tensor(list(dict_node[i]))  
    net_set = torch.tensor(list(dict_net[i]))  

    # Create direct mappings (node -> index, net -> index)
    node_map = {node.item(): idx for idx, node in enumerate(node_set)}
    net_map = {net.item(): idx for idx, net in enumerate(net_set)}

    # Create masks to filter valid edges
    mask_src = torch.isin(edge_index[0], node_set)
    mask_dst = torch.isin(edge_index[1], net_set)
    mask = mask_src & mask_dst  # Keep only valid edges

    indices_to_keep = torch.nonzero(mask, as_tuple=True)[0]  # Indices of valid edges
    filtered_edge_index = edge_index[:, indices_to_keep]  # Filtered edges

    # Apply mapping to get new indices
    mapped_edge_index = torch.tensor([
        [node_map[n.item()] for n in filtered_edge_index[0]],  # Map node indices
        [net_map[n.item()] for n in filtered_edge_index[1]]   # Map net indices
    ])

    # Update dataset with mapped and filtered values
    h_dataset[i]['node', 'to', 'net']["edge_index"] = mapped_edge_index
    h_dataset[i]['node', 'to', 'net']["edge_weight"] = h_dataset[i]['node', 'to', 'net']["edge_weight"][indices_to_keep]
    h_dataset[i]['node', 'to', 'net']["edge_type"] = h_dataset[i]['node', 'to', 'net']["edge_type"][indices_to_keep]

    # Process ('net', 'to', 'node') edges
    edge_index = h_dataset[i]['net', 'to', 'node']["edge_index"]  # (2, num_edges)

    # Create masks for net → node edges
    mask_src = torch.isin(edge_index[0], net_set)
    mask_dst = torch.isin(edge_index[1], node_set)
    mask = mask_src & mask_dst  # Keep only valid edges

    indices_to_keep = torch.nonzero(mask, as_tuple=True)[0]  # Indices of valid edges
    filtered_edge_index = edge_index[:, indices_to_keep]  # Filtered edges

    # Apply mapping to get new indices
    mapped_edge_index = torch.tensor([
        [net_map[n.item()] for n in filtered_edge_index[0]],  # Map net indices
        [node_map[n.item()] for n in filtered_edge_index[1]]   # Map node indices
    ])

    # Update dataset with mapped and filtered values
    h_dataset[i]['net', 'to', 'node']["edge_index"] = mapped_edge_index
    h_dataset[i]['net', 'to', 'node']["edge_weight"] = h_dataset[i]['net', 'to', 'node']["edge_weight"][indices_to_keep]

In [68]:
h_data = h_dataset[0]
if restart:
    model = torch.load(f"{model_type}_{num_layer}_{num_dim}_{vn}_{trans}_model.pt")
elif prediction == 'congestion':
    model = GNN_node(num_layer, num_dim, 2, 2, node_dim = h_data['node'].x.shape[1], net_dim = h_data['net'].x.shape[1], gnn_type=model_type, vn=vn, trans=trans, aggr=aggr, JK="Normal").to(device)
else:
    model = GNN_node(num_layer, num_dim, 1, 1, node_dim = h_data['node'].x.shape[1], net_dim = h_data['net'].x.shape[1], gnn_type=model_type, vn=vn, trans=trans, aggr=aggr, JK="Normal").to(device)

if prediction == 'congestion':
    criterion_node = nn.CrossEntropyLoss()
    criterion_net = nn.CrossEntropyLoss()
else:
    criterion_node = nn.SmoothL1Loss()
    criterion_net = nn.SmoothL1Loss()

optimizer = optim.AdamW(model.parameters(), lr=learning_rate,  weight_decay=0.01)
load_data_indices = [idx for idx in range(len(h_dataset))]
all_train_indices, all_valid_indices, all_test_indices = load_data_indices[:10], load_data_indices[10:11], load_data_indices[11:12]
best_total_val = None

now = datetime.now()
timestamp = now.strftime("%Y-%m-%d_%H-%M-%S")
filepath = f"../results/baselines/{num_epochs}_{num_layer}_{num_dim}_{vn}_{prediction}_baseline.csv"
with open(filepath, 'a') as f:
    if prediction == 'congestion':
        f.write('Epoch,TruePositive,FalsePositive,TrueNegative,FalseNegative,Precision,Recall,Fscore,NodeTrain,NetTrain,NodeValid,NetValid,Time\n')
    else:
        f.write('Epoch,NodeTrain,NetTrain,NodeValid,NetValid,Time\n')

if not test:
    start_time = time.time()
    for epoch in range(num_epochs):
        np.random.shuffle(all_train_indices)
        loss_node_all = 0
        loss_net_all = 0
        val_loss_node_all = 0
        val_loss_net_all = 0
        
        all_train_idx = 0
        for data_idx in tqdm(all_train_indices):
            data = h_dataset[data_idx]
            for inner_data_idx in range(len(data.variant_data_lst)):
                target_node_demand, target_net_demand, target_node_congestion, target_net_congestion, batch, num_vn, vn_node = data.variant_data_lst[inner_data_idx]
                optimizer.zero_grad()
                data.batch = batch
                data.num_vn = num_vn
                data.vn = vn_node
                node_representation, net_representation = model(data, device)
                node_representation = torch.squeeze(node_representation).float()
                net_representation = torch.squeeze(net_representation).float()

                # print(node_representation)
                # print(target_node_demand)

                if prediction == 'congestion':
                    loss_node = criterion_node(node_representation, target_node_congestion.to(device))
                    loss_net = criterion_net(net_representation, target_net_congestion.to(device))
                else:
                    loss_node = criterion_node(node_representation, target_node_demand.to(device))
                    loss_net = criterion_net(net_representation, target_net_demand.to(device))
                
                loss = loss_node + loss_net
                # loss = loss_net
                loss.backward()
                optimizer.step()   
    
                loss_node_all += loss_node.item()
                loss_net_all += loss_net.item()
                all_train_idx += 1
        print('Training Loss Node: ', loss_node_all/all_train_idx, ', Net: ', loss_net_all/all_train_idx)
    
        all_valid_idx = 0


        error_threshold = 10  # Set a threshold for high error instances
        high_error_instances = []
        for data_idx in tqdm(all_valid_indices):
            data = h_dataset[data_idx]
            for inner_data_idx in range(len(data.variant_data_lst)):
                target_node_demand, target_net_demand, target_node_congestion, target_net_congestion, batch, num_vn, vn_node = data.variant_data_lst[inner_data_idx]
                data.batch = batch
                data.num_vn = num_vn
                data.vn = vn_node
                node_representation, net_representation = model(data, device)
                node_representation = torch.squeeze(node_representation)
                net_representation = torch.squeeze(net_representation)

                # Find high error instances
                if prediction == "demand":
                    node_representation_np = node_representation.detach().cpu().numpy()
                    net_representation_np = net_representation.detach().cpu().numpy()

                    target_node_np = target_node_demand.detach().cpu().numpy()
                    target_net_demand_np = target_net_demand.detach().cpu().numpy()

                    node_error = np.abs(node_representation_np - target_node_np)
                    net_error = np.abs(net_representation_np - target_net_demand_np)

                    # Find instances where error exceeds the threshold
                    high_error_nodes = np.where(node_error > error_threshold)[0]
                    high_error_nets = np.where(net_error > error_threshold)[0]

                    for idx in high_error_nodes:
                        high_error_instances.append({
                            "data_idx": data_idx,
                            "instance_type": "node",
                            "instance_index": int(idx),
                            "true_value": float(target_node_np[idx]),
                            "predicted_value": float(node_representation_np[idx]),
                            "error": float(node_error[idx])
                        })

                    for idx in high_error_nets:
                        high_error_instances.append({
                            "data_idx": data_idx,
                            "instance_type": "net",
                            "instance_index": int(idx),
                            "true_value": float(target_net_demand_np[idx]),
                            "predicted_value": float(net_representation_np[idx]),
                            "error": float(net_error[idx])
                        })
                
                # val_loss_node = criterion_node(node_representation, target_node.to(device))
                # val_loss_net = criterion_net(net_representation, target_net.to(device))
                if prediction == 'congestion':
                    val_loss_node = criterion_node(node_representation, target_node_congestion.to(device))
                    val_loss_net = criterion_net(net_representation, target_net_congestion.to(device))
                else:
                    val_loss_node = criterion_node(node_representation, target_node_demand.to(device))
                    val_loss_net = criterion_net(net_representation, target_net_demand.to(device))

                val_loss_node_all +=  val_loss_node.item()
                val_loss_net_all += val_loss_net.item()
                all_valid_idx += 1

                if prediction == 'congestion':
                    outs = node_representation.detach().numpy()
                    pred_vals = np.array([0 if i > j else 1 for i, j in outs])
                    real_vals = target_node_congestion.numpy()
                    tp, fp, tn, fn = 0, 0, 0, 0
                    for i, j in zip(pred_vals, real_vals):
                        if i == 1 and j == 1:
                            tp += 1
                        elif i == 1 and j == 0:
                            fp += 1
                        elif i == 0 and j == 1:
                            fn += 1
                        else:
                            tn += 1
                    p = 0
                    r = 0
                    fsc = 0
                    if tp + fp > 0:
                        p = tp / (tp + fp)
                    if tp + fn > 0:
                        r = tp / (tp + fn)
                    if p + r > 0:
                        fsc = (2 * p * r) / (p + r)

        with open("high_error_instances.json", "w") as json_file:
            json.dump(high_error_instances, json_file, indent=4)

        print(f"Saved {len(high_error_instances)} high-error instances to high_error_instances.json")

        print('Validation Loss Node: ', val_loss_node_all/all_valid_idx, ', Net: ',  val_loss_net_all/all_valid_idx)
        if prediction == 'congestion':
            print(f'Precision: {p}, Recall: {r}, F-score: {fsc}')
            print(tp, fp)
            print(fn, tn)
        print(f'Epoch {epoch+1}, {round(time.time() - start_time, 2)}s\n')
    
        if (best_total_val is None) or ((loss_node_all/all_train_idx) < best_total_val):
            best_total_val = loss_node_all/all_train_idx
            torch.save(model, f"{model_type}_{num_layer}_{num_dim}_{vn}_{trans}_{prediction}.pt")

        with open(filepath, 'a') as f:
            if prediction == 'congestion':
                f.write(f'{epoch+1},{tp},{fp},{tn},{fn},{p},{r},{fsc},')
                f.write(f'{loss_node_all/all_train_idx},{loss_net_all/all_train_idx},{val_loss_node_all/all_valid_idx},{val_loss_net_all/all_valid_idx},{round(time.time()-start_time, 2)}\n')
            else:
                f.write(f'{epoch+1},{loss_node_all/all_train_idx},{loss_net_all/all_train_idx},{val_loss_node_all/all_valid_idx},{val_loss_net_all/all_valid_idx},{round(time.time()-start_time, 2)}\n')
        
else:
    all_test_idx = 0
    test_loss_node_all = 0
    test_loss_net_all = 0
    for data_idx in tqdm(all_test_indices):
        data = h_dataset[data_idx]
        for inner_data_idx in range(len(data.variant_data_lst)):
            target_node, target_net_hpwl, target_net_demand, batch, num_vn, vn_node = data.variant_data_lst[inner_data_idx]
            data.batch = batch
            data.num_vn = num_vn
            data.vn = vn_node
            node_representation, net_representation = model(data, device)
            node_representation = torch.squeeze(node_representation)
            net_representation = torch.squeeze(net_representation)
            
            test_loss_node = criterion_node(node_representation, target_node.to(device))
            test_loss_net = criterion_net(net_representation, target_net_demand.to(device))
            test_loss_node_all +=  test_loss_node.item()
            test_loss_net_all += test_loss_net.item()
            all_test_idx += 1
    print("avg test node demand mse: ", test_loss_node_all/all_test_idx)
    print("avg test net demand mse: ", test_loss_net_all/all_test_idx)

 10%|████▍                                       | 1/10 [00:00<00:01,  4.73it/s]

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


 20%|████████▊                                   | 2/10 [00:00<00:02,  3.69it/s]

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


 30%|█████████████▏                              | 3/10 [00:00<00:01,  4.97it/s]

...

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


 50%|██████████████████████                      | 5/10 [00:01<00:00,  5.40it/s]

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


 50%|██████████████████████                      | 5/10 [00:01<00:01,  3.42it/s]


KeyboardInterrupt: 

In [17]:
h_dataset[0]["variant_data_lst"]

[(tensor([-0.7202, -1.1645, -1.3423,  ...,  2.9231,  2.4788,  2.0345]),
  tensor([-3.1723, -2.7618, -2.4110,  ...,  2.3874,  2.5696,  2.5598]),
  tensor([1, 0, 0,  ..., 1, 1, 1]),
  tensor([0, 0, 0,  ..., 1, 1, 1]),
  tensor([55, 36,  4,  ...,  8, 58, 14]),
  61,
  tensor([[2.0393e+02, 5.9516e-04, 2.7430e-05,  ..., 1.8620e+03, 2.5430e+03,
           3.2120e+03],
          [2.2254e+02, 7.8079e-04, 2.9264e-06,  ..., 1.4370e+03, 1.4380e+03,
           1.9100e+03],
          [1.8899e+02, 6.8883e-04, 0.0000e+00,  ..., 7.2100e+02, 1.4220e+03,
           1.7200e+03],
          ...,
          [1.4226e+02, 1.0015e-03, 2.8025e-06,  ..., 5.0400e+02, 5.0400e+02,
           4.2200e+02],
          [1.1457e+02, 1.2394e-03, 0.0000e+00,  ..., 1.1630e+03, 2.0820e+03,
           3.2370e+03],
          [1.6163e+02, 9.6175e-04, 4.2038e-06,  ..., 6.2070e+03, 7.2240e+03,
           7.3940e+03]]))]

In [14]:
dict_node[0]

[102746,
 36342,
 308905,
 315473,
 283727,
 253669,
 397187,
 150507,
 282023,
 386153,
 92292,
 375520,
 343929,
 238563,
 168205,
 474162,
 332233,
 401978,
 386562,
 354178,
 177493,
 396947,
 477604,
 35332,
 314168,
 149517,
 93689,
 69065,
 249735,
 466034,
 252128,
 308543,
 271366,
 146029,
 275384,
 332224,
 313796,
 465179,
 278630,
 203176,
 282217,
 152683,
 314127,
 108869,
 9512,
 39736,
 441750,
 351990,
 25708,
 443437,
 477092,
 458000,
 390836,
 34856,
 438642,
 313414,
 78284,
 468979,
 461706,
 414639,
 239436,
 406583,
 276608,
 459721,
 86353,
 470446,
 84093,
 150446,
 260062,
 56843,
 381576,
 270766,
 489445,
 104897,
 142400,
 83883,
 462687,
 458514,
 111495,
 334092,
 335690,
 82117,
 458672,
 180799,
 95170,
 235837,
 151349,
 276407,
 252133,
 436475,
 85467,
 78336,
 17602,
 471981,
 385841,
 89232,
 114972,
 435665,
 456323,
 162625,
 396297,
 307472,
 464407,
 32008,
 405420,
 26524,
 383823,
 398502,
 253698,
 64533,
 68886,
 354386,
 475148,
 343541,

In [ ]:
h_dataset.copy()

In [ ]:
h_dataset[0]["variant_data_lst"]

In [18]:
h_dataset

[HeteroData(
   design_name='superblue_19',
   num_instances=16730,
   variant_data_lst=[1],
   node={ x=[16730, 45] },
   net={ x=[51037, 11] },
   (node, to, net)={
     edge_index=[2, 5460],
     edge_weight=[5460],
     edge_type=[5460],
   },
   (net, to, node)={
     edge_index=[2, 5460],
     edge_weight=[5460],
   }
 ),
 HeteroData(
   design_name='superblue_18',
   num_instances=38325,
   variant_data_lst=[1],
   node={ x=[38325, 45] },
   net={ x=[43709, 11] },
   (node, to, net)={
     edge_index=[2, 21551],
     edge_weight=[21551],
     edge_type=[21551],
   },
   (net, to, node)={
     edge_index=[2, 21551],
     edge_weight=[21551],
   }
 ),
 HeteroData(
   design_name='superblue_16',
   num_instances=25771,
   variant_data_lst=[1],
   node={ x=[25771, 45] },
   net={ x=[47088, 11] },
   (node, to, net)={
     edge_index=[2, 6352],
     edge_weight=[6352],
     edge_type=[6352],
   },
   (net, to, node)={
     edge_index=[2, 6352],
     edge_weight=[6352],
   }
 ),
 Hete

In [ ]:
h_dataset[0]["variant_data_lst"][0][0][dict_node[0]]

In [ ]:
h_dataset[0]["batch"]

In [ ]:
count = (node_demand < -2).sum().item() + (net_demand < -2).sum().item()
count

In [ ]:
count = (node_demand > 3).sum().item() + (net_demand > 3).sum().item()
count

In [ ]:
len(list4)

In [ ]:
h_dataset[i]["variant_data_lst"][0]